<a href="https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/12_big_data_beyond_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[⬅ Back to the workshop index](https://github.com/lorenzkap/ML2026#-the-notebooks) · **Notebook 12 of the course**


# 🩺 Notebook 12 — When your data is too big for pandas *(bonus, reference)*

> Our teaching file is tiny (~10 MB). Real projects can be **gigabytes to terabytes**. This short
> reference explains **when pandas stops working** and **what to reach for instead** — plus where the
> really big datasets come from. It's for *information*: skim it, keep it as a cheat-sheet.

**In one line:** pandas loads everything into **RAM**, so it breaks when a dataset (× a few) no longer
fits in memory. The fixes are: *load less*, *use a better file format*, *process in chunks*, or *switch to
an out-of-core / parallel tool*.

**⏱️ Time:** ≈15 min · **Level:** reference / take‑home

### ⚙️ Run me first

In [1]:
# === ⚙️  Workshop setup — run this cell first ===============================
# Works in Google Colab and in local Jupyter. Installs anything missing, sets a
# clean plotting style, and gives you helpers to load the data.
# (This cell is identical in every notebook of the course.)
import importlib.util, subprocess, sys, os, random, warnings
warnings.filterwarnings("ignore")

# --- reproducibility: everyone in the room gets the same numbers ---------------
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)

# 👩‍🏫 INSTRUCTORS: put a direct-download link to the workshop data here (or set the
# WORKSHOP_DATA_URL environment variable) and participants never upload anything.
# It must be a folder-style base URL that serves the CSVs by name, e.g.
#     "https://your-institution.example/sepsis_workshop/"
# Leave it empty ("") to use the upload flow instead. The data is NOT in the public
# GitHub repo on purpose — it is derived from MIMIC-IV and covered by a PhysioNet DUA.
WORKSHOP_DATA_URL = os.environ.get("WORKSHOP_DATA_URL", "")

def _ensure(pkgs):
    missing = [pip for mod, pip in pkgs.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing])
_ensure({"numpy":"numpy","pandas":"pandas","sklearn":"scikit-learn",
         "matplotlib":"matplotlib","seaborn":"seaborn","shap":"shap","xgboost":"xgboost"})

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(RANDOM_STATE)          # seeds the legacy global np.random.* calls
RNG = np.random.default_rng(RANDOM_STATE)   # the modern generator — use this one
pd.set_option("display.max_columns", 120); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5); plt.rcParams["figure.dpi"] = 110

# Every model, split and resample in this course passes random_state=RANDOM_STATE, so
# your numbers should match your neighbour's exactly. (Different library *versions* can
# still shift the last decimal — that is normal and not a mistake on your part.)

# --- data loading: works locally AND remembers your upload across notebooks -----
_CACHE = {"dir": "unset"}   # memo so we only touch Google Drive once per session

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _drive_cache():
    """In Google Colab, mount Drive ONCE and return a persistent folder. A file you
    upload in one notebook is saved here, so every other notebook opens it automatically
    — no re-uploading. Returns None outside Colab, or if you decline to connect Drive."""
    if _CACHE["dir"] != "unset":
        return _CACHE["dir"]
    result = None
    if _in_colab():
        try:
            from google.colab import drive
            if not os.path.ismount("/content/drive"):
                drive.mount("/content/drive")
            result = "/content/drive/MyDrive/sepsis_workshop_data"
            os.makedirs(result, exist_ok=True)
        except Exception:
            result = None
    _CACHE["dir"] = result
    return result

def _find(name):
    paths = [name, f"data/{name}", f"../data/{name}", f"workshop/data/{name}"]
    cache = _drive_cache()
    if cache:
        paths.append(os.path.join(cache, name))
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _try_download(name):
    """If the instructor configured WORKSHOP_DATA_URL, fetch the file from there."""
    if not WORKSHOP_DATA_URL:
        return None
    import urllib.request
    url = WORKSHOP_DATA_URL.rstrip("/") + "/" + name
    try:
        print(f"⬇  downloading {name} …")
        urllib.request.urlretrieve(url, name)
        return name
    except Exception as e:
        print(f"  (download failed: {e})")
        return None

def _cache_to_drive(name, data):
    cache = _drive_cache()
    if cache:
        dest = os.path.join(cache, name)
        data.to_csv(dest, index=False)
        print(f"  💾 saved to Google Drive ({dest}) — no need to fetch it again.")

def load_csv(name):
    """Load a data CSV. Searches local folders, (in Colab) your Google-Drive cache, and
    the instructor's download link. If none of those work it asks you to upload the file
    ONCE, then saves a copy to Drive so the other notebooks find it automatically."""
    p = _find(name)
    if p:
        print(f"✓ loaded {p}")
        return pd.read_csv(p)
    p = _try_download(name)
    if p:
        data = pd.read_csv(p)
        print(f"✓ loaded {name}")
        _cache_to_drive(name, data)
        return data
    try:
        from google.colab import files
        print(f"⤴  Upload {name} just once — I'll save it so the other notebooks open it automatically:")
        up = files.upload()
        fname = list(up.keys())[0]
        data = pd.read_csv(fname)
        _cache_to_drive(name, data)
        return data
    except Exception:
        raise FileNotFoundError(
            f"Could not find {name}. Ask your instructor for the workshop CSVs, then either "
            f"upload it when prompted, or put it next to this notebook / in a data/ folder."
        )


## 🧱 When does pandas hit a wall?

A pandas `DataFrame` lives **entirely in RAM**, and operations often make temporary copies — a good rule
of thumb is you need **≈ 5–10 × the raw file size** in free memory. So a 4 GB CSV can exhaust a 16–32 GB
laptop. Warning signs: `MemoryError`, the machine **swapping** to disk, or everything crawling.

In [2]:
# How much memory does our (small) table really use? deep=True counts the strings too.
df = load_csv("sepsis_timeseries.csv")
mem_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"{len(df):,} rows × {df.shape[1]} cols  →  {mem_mb:.1f} MB in RAM")
print(f"Rule of thumb: comfortably handle a file if you have ~5–10× its size in free RAM.")
print(f"So on a 16 GB laptop, plain pandas is happy up to roughly ~1–3 GB of CSV.")

✓ loaded data/sepsis_timeseries.csv
37,704 rows × 53 cols  →  15.7 MB in RAM
Rule of thumb: comfortably handle a file if you have ~5–10× its size in free RAM.
So on a 16 GB laptop, plain pandas is happy up to roughly ~1–3 GB of CSV.


## 🔧 First — squeeze more out of pandas (often enough!)

Before switching tools, three cheap wins:

**1. Load less / use smaller dtypes.** Read only the columns you need (`usecols`), and **downcast**:
`float64→float32`, `int64→int32`, and repeated strings → `category`. Often halves memory.

In [3]:
small = df.copy()
for c in small.select_dtypes("float64"): small[c] = pd.to_numeric(small[c], downcast="float")
for c in small.select_dtypes("int64"):   small[c] = pd.to_numeric(small[c], downcast="integer")
small["gender"] = small["gender"].astype("category")
after = small.memory_usage(deep=True).sum() / 1e6
print(f"memory: {mem_mb:.1f} MB  →  {after:.1f} MB  ({100*(1-after/mem_mb):.0f}% smaller) just from dtypes")

memory: 15.7 MB  →  7.2 MB  (54% smaller) just from dtypes


**2. Use a columnar format — Parquet, not CSV.** Parquet is compressed, typed, and reads only the
columns you ask for. It's usually **much smaller and far faster** than CSV (and preserves dtypes).

In [4]:
import os
df.to_parquet("_demo.parquet")             # needs pyarrow (installed in Colab)
df.to_csv("_demo.csv", index=False)
csv_mb = os.path.getsize("_demo.csv") / 1e6
pq_mb  = os.path.getsize("_demo.parquet") / 1e6
print(f"CSV: {csv_mb:.1f} MB   Parquet: {pq_mb:.1f} MB   →  {csv_mb/pq_mb:.1f}× smaller")
# read back just two columns — Parquet doesn't touch the rest
two = pd.read_parquet("_demo.parquet", columns=["icustayid", "Arterial_lactate"])
print("read only 2 columns from Parquet:", two.shape)
os.remove("_demo.parquet"); os.remove("_demo.csv")

CSV: 9.5 MB   Parquet: 2.5 MB   →  3.8× smaller
read only 2 columns from Parquet: (37704, 2)


**3. Stream it in chunks.** If a CSV won't fit, read it in pieces with `chunksize` and aggregate as
you go — constant memory, whatever the file size.

In [5]:
df.to_csv("_big.csv", index=False)
# Compute a per-patient max lactate WITHOUT ever holding the whole file in memory:
running = None
for chunk in pd.read_csv("_big.csv", usecols=["icustayid", "Arterial_lactate"], chunksize=5000):
    part = chunk.groupby("icustayid")["Arterial_lactate"].max()
    running = part if running is None else pd.concat([running, part]).groupby(level=0).max()
print(f"processed the file in chunks → {len(running):,} patients, no full load")
os.remove("_big.csv")

processed the file in chunks → 1,696 patients, no full load


## 🚀 When pandas truly isn't enough — the alternatives

| Tool | What it is | Reach for it when |
|---|---|---|
| **Polars** | pandas-like DataFrame, multi-threaded, **lazy** + streaming (Rust) | You want a big speed/memory win with familiar code; single machine, up to *bigger-than-RAM* via streaming |
| **DuckDB** | in-process **SQL** engine over CSV/Parquet, out-of-core | You think in SQL and want to query files **larger than RAM** on one machine (integrates with pandas/Arrow) |
| **Dask** | parallel, partitioned **pandas API** | You want the pandas API to scale across cores or a cluster / bigger-than-RAM |
| **Apache Spark (PySpark)** | distributed cluster compute | **Terabytes+**, many machines, production data engineering |
| **Vaex** | out-of-core DataFrames for exploration/plots | Interactively explore/visualise **billions** of rows |
| **A database / warehouse** | Postgres, BigQuery, Snowflake… | Push the heavy joins/aggregations to where the data already lives |

A few one-liners (install first, e.g. `pip install polars duckdb`):
```python
# Polars — lazy scan, filter, aggregate, only materialise the result:
import polars as pl
pl.scan_parquet("big.parquet").filter(pl.col("SOFA") > 6).group_by("icustayid").agg(pl.col("Arterial_lactate").max()).collect()

# DuckDB — SQL directly on files bigger than RAM (no full load):
import duckdb
duckdb.sql("SELECT icustayid, MAX(Arterial_lactate) FROM 'big.parquet' GROUP BY icustayid").df()

# Dask — pandas API, partitioned across cores/cluster:
import dask.dataframe as dd
dd.read_parquet("big/*.parquet").groupby("icustayid").Arterial_lactate.max().compute()
```

## 🧭 A rough decision guide

| Data size (single machine, ~16 GB RAM) | Use |
|---|---|
| Fits in RAM (≲ 1–3 GB CSV) | **pandas** (or **Polars** for speed) |
| Bigger than RAM, one machine (~GBs–100s GB) | **DuckDB** or **Polars (streaming)**; chunked pandas; convert to **Parquet** |
| Won't fit + needs many cores / a cluster | **Dask** |
| Terabytes, many machines, production | **Spark**, or a **cloud warehouse** (BigQuery/Snowflake) |

> 🧠 **Takeaway.** 90% of "big data" problems on a laptop are solved by **Parquet + dtype downcasting +
> chunking**, or by switching to **Polars/DuckDB** — no cluster needed. Reach for Spark only when you
> genuinely have many machines' worth of data.

## 🗂️ Where do the really big datasets come from? *(sources)*

**Large public ICU / EHR / health datasets** (most need credentialing + a data-use agreement):
- **MIMIC-IV** (full) — ~40k+ ICU stays; the source of *this* course's sample · [physionet.org](https://physionet.org/content/mimiciv/)
- **eICU-CRD** — 200k+ ICU stays, multi-centre US · **HiRID**, **AmsterdamUMCdb** — high-resolution ICU
- **UK Biobank** — 500k participants, genetics + imaging + EHR · **All of Us** (NIH) — 1M+ participants
- **PhysioNet** — dozens of waveform / clinical datasets · **MIMIC-CXR** — 377k chest X-rays (imaging = huge)
- **SEER** — cancer registry · **OMOP CDM / OHDSI** — federated EHR across institutions
- Your own hospital's **data warehouse** (often the biggest — and messiest — of all)

These arrive as **CSV, Parquet, HDF5, Arrow, or in a database** — for anything above a few GB, prefer
Parquet/database access over one giant CSV.

## 📚 Further reading
- pandas — *Scaling to larger datasets* (official guide) · **Polars**, **DuckDB**, **Dask**, **Spark**,
  **Vaex** documentation · Apache **Arrow** / **Parquet** (the columnar formats underneath most of these).

## ✅ Recap
- pandas is a **RAM** tool: great to ~1–3 GB on a laptop, then it struggles.
- Stretch it with **`usecols` + dtype downcasting**, **Parquet**, and **`chunksize`** streaming.
- Beyond that: **Polars / DuckDB** (one machine, out-of-core), **Dask** (parallel pandas), **Spark** (cluster).
- Store big data as **Parquet**, not CSV — and let a **database/warehouse** do the heavy lifting when you can.